<a href="https://colab.research.google.com/github/Mohsin-22/Assignment6-100_Gen-Ai_cohort-/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 6: Retrieval-Augmented Generation (RAG) - Machine Learning Knowledge Assistant

In [ ]:
# Installing required Libraries
!pip install faiss-cpu chromadb tiktoken sentence-transformers transformers
!pip install pypdf


In [ ]:
# Loading pdf and extracting the text
from pypdf import PdfReader

reader = PdfReader("/content/intro-to-ml.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print("Total Pages:", len(reader.pages))
print("Sample Text:", text[:1000])

In [ ]:
import re

def clean_text(text):
    # Fix spaced letters like "P y t h o n"
    text = re.sub(r'\b(\w)\s+(?=\w\b)', r'\1', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)

    # Fix HTML entities
    text = text.replace("&amp;", "&")

    return text

text = clean_text(text)

print("Cleaned Sample:", text[:500])

In [ ]:
# Exploring the Document
print("Total Characters:", len(text))

In [ ]:
!pip install -U langchain langchain-text-splitters

In [ ]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Define splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Splitting the text
chunks = splitter.split_text(text)

# Output
print("Total Chunks:", len(chunks))
print("\nSample Chunk:\n", chunks[0][:500])



In [ ]:
# Embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# Create Embeddings
embeddings = model.encode(chunks, normalize_embeddings=True)

In [ ]:
#Store in FAISS

import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(np.array(embeddings))

In [ ]:
# Similarity Search
def retrieve(query, k=5):
    query_vec = model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_vec, k)

    results = []
    for i in indices[0]:
        chunk = chunks[i]

        # Filter noisy chunks
        if len(chunk) > 100 and "print(" not in chunk and "In[" not in chunk:
            results.append(chunk)

    return results[:3]

In [ ]:
#Example Retrieval
query = "What is overfitting?"
results = retrieve(query)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n", r)

In [ ]:
for k in [3, 5, 7]:
    print(f"\nTop {k} results:\n")
    results = retrieve("What is supervised learning?", k=k)
    for r in results:
        print("-", r[:200])

In [ ]:
#Prompt Design
def build_prompt(query, context_chunks):
    context = "\n\n".join(context_chunks)

    return f"""
You are a helpful AI assistant.

Answer the question using ONLY the given context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {query}

Answer:
"""


In [ ]:
# LLM
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


In [ ]:
#  Generate Answer
def generate_answer(query):
    retrieved_chunks = retrieve(query)

    context = "\n".join(retrieved_chunks)
    context = context[:1200]

    prompt = f"""
You are a Machine Learning expert.

Using ONLY the context below, write a clear and complete answer in 2–3 sentences.

Context:
{context}

Question: {query}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=150,
        min_length=40,
        repetition_penalty=2.0,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
#Example Q&A
questions = [
    "What is Retrival augmented generation?",
    "Explain supervised learning",
    "What is k nearest neighbors?",
    "What are Ensemble Models?"
]

for q in questions:
    print("\nQuestion:", q)
    print("Answer:", generate_answer(q))

In [ ]:
# End-to-End Application
def rag_chatbot(query):
    retrieved_chunks = retrieve(query, k=5)
    answer = generate_answer(query)

    print("\n Retrieved Context:")
    for c in retrieved_chunks[:2]:
        print("-", c[:200])

    print("\n\n Answer:")
    print(answer)

In [ ]:
#Test Application
rag_chatbot("What is Supervised Learning in machine learning?")